# 1. Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import pandas as pd
import re
import time
from datetime import datetime

# 2. Define Settings

In [ ]:
BASE_URL = 'https://www.fcc.gov/documents?field_edoc_doctype_target_id%5B0%5D=69&field_released_date_value=&exposed_from_date=2010-01-01&exposed_to_date='

FCC_HOME = 'https://www.fcc.gov'

PAGE_START = 0
PAGE_END = 4

OUTPUT_FOLDER = Path(f'fcc_news_release_documents_page_{PAGE_START}-{PAGE_END}')
OUTPUT_FOLDER.mkdir(exist_ok=True)

LINKS_FILE = OUTPUT_FOLDER / 'fcc_news_release_links.csv'
METADATA_FILE = OUTPUT_FOLDER / 'fcc_news_release_metadata.csv'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (academic research)',
    'Accept-Language': 'en',
}

TIMEOUT = 60
DELAY = 1

# 3. Define Helper Functions

In [ ]:
def create_session(headers=HEADERS):
    """
    Create a requests session with browser-like headers.
    """
    session = requests.Session()
    session.headers.update(headers)
    return session

In [ ]:
def get_soup(session, url, timeout=TIMEOUT):
    """
    Download a webpage and return a BeautifulSoup object.
    """
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

In [ ]:
def build_search_url(page):
    """
    Build the FCC search result URL for a given page number.
    """
    return f'{BASE_URL}&page={page}'

In [ ]:
def clean_text(text):
    """
    Clean extra spaces from extracted text.
    """
    if text is None:
        return None
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
def clean_filename(title, max_length=150):
    """
    Convert a document title into a safe file name.
    """
    clean_title = re.sub(r'[\\/*?:"<>|]', '', title)
    clean_title = re.sub(r'\s+', '_', clean_title)
    clean_title = clean_title.strip('_')

    if not clean_title:
        clean_title = 'untitled'

    return clean_title[:max_length]

In [ ]:
def format_date_for_filename(date_text):
    """
    Convert FCC date text like 'Apr 29, 2026' into '260429'.
    """
    if not date_text:
        return 'unknown_date'

    date_text = date_text.strip()

    for date_format in ['%b %d, %Y', '%B %d, %Y']:
        try:
            date_object = datetime.strptime(date_text, date_format)
            return date_object.strftime('%y%m%d')
        except ValueError:
            continue

    return 'unknown_date'

In [ ]:
def get_one_text(soup, selector):
    """
    Extract one text item from a CSS selector.
    """
    tag = soup.select_one(selector)
    if tag is None:
        return None
    return clean_text(tag.get_text(' ', strip=True))

In [ ]:
def get_many_texts(soup, selector):
    """
    Extract multiple text items from a CSS selector.
    """
    texts = []

    for tag in soup.select(selector):
        text = clean_text(tag.get_text(' ', strip=True))
        if text:
            texts.append(text)

    return texts

In [ ]:
def get_date_text(soup, selector):
    """
    Extract date text from FCC date fields.
    """
    tag = soup.select_one(selector)

    if tag is None:
        return None

    time_tag = tag.select_one('time')

    if time_tag is None:
        return None

    return clean_text(time_tag.get_text(' ', strip=True))

In [ ]:
def normalize_label(text):
    """
    Normalize document captions and document types for comparison.
    Example: 'News Release:' -> 'news release'
    """
    text = clean_text(text) or ''
    text = re.sub(r'\s*:\s*$', '', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return text.strip()

In [ ]:
def extract_news_release_txt_urls(soup):
    """
    Extract the main News Release txt file from an FCC document page.
    Ignore supplementary files such as statements or related documents.
    """
    selected_txt_urls = []
    all_txt_urls = []

    for block in soup.select('span.edoc-document'):
        caption_tag = block.select_one('.document-caption')
        txt_tag = block.select_one('a.document-link.txt[href]')

        if txt_tag is None:
            continue

        txt_url = urljoin(FCC_HOME, txt_tag.get('href'))
        all_txt_urls.append(txt_url)

        if caption_tag is None:
            continue

        caption = clean_text(caption_tag.get_text(' ', strip=True))
        caption_norm = normalize_label(caption)

        if caption_norm == 'news release':
            selected_txt_urls.append(txt_url)

    return selected_txt_urls, all_txt_urls

In [ ]:
def extract_document_metadata(session, document_url):
    """
    Open one FCC document webpage and extract metadata and txt file URLs.
    """
    soup = get_soup(session, document_url)

    full_title = get_one_text(soup, '.edoc__full-title .field__item')
    page_title = get_one_text(soup, 'h1.display-4 span') or get_one_text(soup, 'h1')

    document_type = get_one_text(soup, '.edoc__doctype .field__item')
    bureaus = get_many_texts(soup, '.edoc__field-bureau-office-category .field__item')

    description = get_one_text(soup, '.edoc__short-desc .field__item')

    released_on = get_date_text(soup, '.edoc__release-dt')
    issued_on = get_date_text(soup, '.edoc__publish-dt')
    adopted = get_date_text(soup, '.edoc__adopted-dt')

    selected_txt_urls, all_txt_urls = extract_news_release_txt_urls(soup)

    selected_txt_urls = list(dict.fromkeys(selected_txt_urls))

    metadata = {
        'page_title': page_title,
        'full_title': full_title,
        'document_type': document_type,
        'bureaus': '; '.join(bureaus) if bureaus else None,
        'description': description,
        'webpage_url': document_url,
        'selected_txt_urls': selected_txt_urls,
        'selected_txt_count': len(selected_txt_urls),
        'all_txt_urls': all_txt_urls,
        'all_txt_count': len(all_txt_urls),
        'released_on': released_on,
        'issued_on': issued_on,
        'adopted': adopted,
        'downloaded_files': [],
        'download_status': None,
        'error_message': None
    }

    return metadata

In [ ]:
def extract_document_links_from_soup(soup):
    """
    Extract document titles and document page links from one FCC search result page.
    """
    documents = []

    for a in soup.select('div.headline-title a.title'):
        title = a.get_text(strip=True)
        href = a.get('href')

        if not href:
            continue

        link = urljoin(FCC_HOME, href)

        documents.append({
            'title': title,
            'webpage_url': link
        })

    return documents

In [ ]:
def collect_document_links(session, start_page=PAGE_START, end_page=PAGE_END, delay=DELAY):
    """
    Go through FCC search result pages and collect document webpage links
    and save document links to CSV.
    """
    document_links = []

    for page in range(start_page, end_page + 1):
        # print('=' * 60)
        # print(f'Search page {page}')
        print(page, end=' | ')

        page_url = build_search_url(page)

        try:
            soup = get_soup(session, page_url)
            page_documents = extract_document_links_from_soup(soup)

            document_links.extend(page_documents)

            # for doc in page_documents:
            #     print(doc['title'], '>>>', doc['webpage_url'])

            # print(f'Documents found on page {page}: {len(page_documents)}')

        except requests.RequestException as e:
            print(f'>>> Failed to collect page {page}: {e}')

        time.sleep(delay)

    print('\nTotal number of document links:', len(document_links))

    links_df = pd.DataFrame(document_links)
    links_df.to_csv(LINKS_FILE, index=False, encoding='utf-8-sig')

    return document_links

In [ ]:
def save_txt_file(session, txt_url, file_path, timeout=TIMEOUT):
    """
    Download one txt file and save it.
    """
    response = session.get(txt_url, timeout=timeout)
    response.raise_for_status()
    
    file_path.write_text(response.text, encoding='utf-8')

In [ ]:
def save_df_for_csv(df, list_columns):
    """
    Save dataframe in a csv-friendly format.
    """

    df_for_csv = df.copy()
    
    for col in list_columns:
        if col in df_for_csv.columns:
            df_for_csv[col] = df_for_csv[col].apply(
                lambda x: ' | '.join(map(str, x)) if isinstance(x, list) else x
            )
    
    df_for_csv.to_csv(
        METADATA_FILE,
        index=False,
        encoding='utf-8-sig'
    )

In [ ]:
def download_news_release_documents(
    session,
    document_links,
    output_folder=OUTPUT_FOLDER,
    delay=DELAY,
    save_csv=True
):
    """
    For each FCC News Release page:
    1. collect metadata,
    2. select the txt file matching the document type,
    3. download the selected txt file,
    4. store metadata in a dataframe.
    """
    output_folder.mkdir(exist_ok=True)

    metadata_records = []

    for i, doc in enumerate(document_links, start=1):
        # print('=' * 60)
        # print(f"Document_{i}: {doc['title']}")
        print(i, end=' | ')

        try:
            metadata = extract_document_metadata(
                session=session,
                document_url=doc['webpage_url']
            )

            title_for_filename = (
                doc['title']
                or f'document_{i}'
            )

            date_for_filename = (
                format_date_for_filename(metadata['released_on'])
                if metadata['released_on']
                else format_date_for_filename(metadata['issued_on'])
            )

            selected_txt_urls = metadata['selected_txt_urls']

            if not selected_txt_urls:
                print(f"{doc['title']} >>> No matching txt file found.")
                metadata['download_status'] = 'skipped_no_matching_txt'
                metadata_records.append(metadata)
                continue

            downloaded_files = []

            for j, txt_url in enumerate(selected_txt_urls, start=1):
                clean_title = clean_filename(title_for_filename)

                if len(selected_txt_urls) == 1:
                    file_name = f'{date_for_filename}_{clean_title}.txt'
                else:
                    file_name = f'{date_for_filename}_{clean_title}_{j}.txt'

                file_path = output_folder / file_name

                save_txt_file(session, txt_url, file_path)

                downloaded_files.append(str(file_path))
                # print('Saved:', file_path)

            metadata['downloaded_files'] = downloaded_files
            metadata['download_status'] = 'downloaded'

            metadata_records.append(metadata)

        except requests.RequestException as e:
            print(f"{doc['title']} >>> Request failed: {e}")

            metadata_records.append({
                'page_title': doc.get('title'),
                'full_title': None,
                'document_type': None,
                'bureaus': None,
                'description': None,
                'webpage_url': doc.get('webpage_url'),
                'selected_txt_urls': [],
                'selected_txt_count': 0,
                'all_txt_urls': [],
                'all_txt_count': 0,
                'released_on': None,
                'issued_on': None,
                'adopted': None,
                'downloaded_files': [],
                'download_status': 'failed_request',
                'error_message': str(e)
            })

        time.sleep(delay)

    metadata_df = pd.DataFrame(metadata_records)

    if save_csv:

        list_columns = ['selected_txt_urls', 'all_txt_urls', 'downloaded_files']

        save_df_for_csv(metadata_df, list_columns)

    print('\n')
    print('<Download summary>')
    print('Downloaded:', (metadata_df['download_status'] == 'downloaded').sum())
    print('Skipped:', (metadata_df['download_status'] == 'skipped_no_matching_txt').sum())
    print('Failed:', (metadata_df['download_status'] == 'failed_request').sum())
    
    if save_csv:
        print('Metadata saved to:', METADATA_FILE)

    return metadata_df

In [ ]:
def retry_failed_downloads(
    metadata_df,
    session,
    output_folder=OUTPUT_FOLDER,
    max_rounds=5,
    delay=DELAY
):
    """
    Retry documents whose download_status is 'failed_request'.

    The function repeats retry rounds until:
    1. no failed_request rows remain, or
    2. max_rounds is reached.

    Set max_rounds=None to run indefinitely (Not recommended!).
    """
    metadata_df = metadata_df.copy()

    round_number = 0

    while True:
        failed_mask = metadata_df['download_status'] == 'failed_request'
        failed_indices = metadata_df.index[failed_mask].tolist()

        if not failed_indices:
            print('\nNo failed downloads remaining.')
            break

        if max_rounds is not None and round_number >= max_rounds:
            print(f'\nStopped after {max_rounds} retry rounds.')
            print('Remaining failed downloads:', len(failed_indices))
            break

        round_number += 1

        print('\n' + '=' * 60)
        print(f'Retry round {round_number}')
        print('Failed documents to retry:', len(failed_indices))
        print('=' * 60)

        failed_document_links = []

        for idx in failed_indices:
            row = metadata_df.loc[idx]

            webpage_url = row.get('webpage_url')

            title = row.get('page_title')

            if not webpage_url:
                metadata_df.at[idx, 'download_status'] = 'failed_request'
                metadata_df.at[idx, 'error_message'] = 'Missing webpage_url'
                continue

            failed_document_links.append({
                'title': title,
                'webpage_url': webpage_url,
                'original_index': idx
            })

        retry_df = download_news_release_documents(
            session=session,
            document_links=failed_document_links,
            output_folder=output_folder,
            delay=delay,
            save_csv=False
        )

        for old_idx, (_, new_row) in zip(
            [doc['original_index'] for doc in failed_document_links],
            retry_df.iterrows()
        ):
            for col, value in new_row.items():
                metadata_df.at[old_idx, col] = value

        list_columns = ['selected_txt_urls', 'all_txt_urls', 'downloaded_files']

        save_df_for_csv(metadata_df, list_columns)

        print('\n')
        print('<Retry summary>')
        print('Downloaded:', (metadata_df['download_status'] == 'downloaded').sum())
        print('Skipped:', (metadata_df['download_status'] == 'skipped_no_matching_txt').sum())
        print('Still failed:', (metadata_df['download_status'] == 'failed_request').sum())
        print('Metadata saved to:', METADATA_FILE)

    return metadata_df

# 4. Collect Document Links

In [ ]:
session = create_session()

document_links = collect_document_links(
    session=session,
    start_page=PAGE_START,
    end_page=PAGE_END,
    delay=DELAY
)

# 5. Download txt files and collect metadata

In [ ]:
metadata_df = download_news_release_documents(
    session=session,
    document_links=document_links,
    output_folder=OUTPUT_FOLDER,
    delay=DELAY
)

# 6. Retry Failed Document Downloads

In [ ]:
metadata_df = retry_failed_downloads(
    metadata_df,
    session,
    output_folder=OUTPUT_FOLDER,
    max_rounds=5,
    delay=DELAY
)